# spaCy Tokenization Deep Dive
Tokenization is the process of splitting text into meaningful units (tokens). spaCy's tokenizer works in two stages:
1. **Whitespace split** — splits on spaces first
2. **Prefix/suffix/infix rules** — further splits punctuation, currency symbols, etc.

This produces linguistically meaningful tokens rather than naive character-based splits.

### `spacy.blank("en")` vs `spacy.load("en_core_web_sm")`
| | `spacy.blank()` | `spacy.load()` |
|---|---|---|
| Pipeline components | None (tokenizer only) | tok2vec, tagger, parser, NER, lemmatizer |
| Use case | Fast tokenization, custom pipelines | Full NLP analysis |
| Model size | Negligible | ~12 MB (sm), ~43 MB (md) |

Notice that `$` is separated from `2` in the output — spaCy treats currency symbols as standalone tokens.

In [31]:
import spacy

In [32]:
nlp = spacy.blank("en")

doc = nlp("Dr. Strange loves pav bhaji of mumbai as it costs only 2$ per plate.")
for token in doc:
    print(token.text)

Dr.
Strange
loves
pav
bhaji
of
mumbai
as
it
costs
only
2
$
per
plate
.


### `spacy.blank("en")` — Tokenizer Only
Creates a blank English pipeline with **only the tokenizer** — no POS tagger, no NER, no parser. Useful when you need fast tokenization without the overhead of a full model.

### `spacy.load("en_core_web_sm")` — Full Pre-trained Pipeline
Loads a small English model with all pipeline components active: **tok2vec → tagger → parser → attribute_ruler → lemmatizer → NER**. Use this when you need POS tags, lemmas, or named entities.

###  The `Doc` Object & Token Indexing
A `Doc` is a sequence of `Token` objects. You can access individual tokens by integer index (`doc[0]`, `doc[1]`). Each `Token` carries rich linguistic metadata — inspect all available attributes with `dir(token)`.

In [33]:
doc[0]

Dr.

In [34]:
token = doc[1]
token.text

'Strange'

`dir(token)` reveals all attributes and methods on a `Token`. Key ones to remember: `.text`, `.lemma_`, `.pos_`, `.is_alpha`, `.is_stop`, `.like_email`, `.like_url`, `.is_currency`.

In [35]:
dir(token)

['_',
 '__bytes__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__ne__',
 '__new__',
 '__pyx_vtable__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__unicode__',
 'ancestors',
 'check_flag',
 'children',
 'cluster',
 'conjuncts',
 'dep',
 'dep_',
 'doc',
 'ent_id',
 'ent_id_',
 'ent_iob',
 'ent_iob_',
 'ent_kb_id',
 'ent_kb_id_',
 'ent_type',
 'ent_type_',
 'get_extension',
 'has_dep',
 'has_extension',
 'has_head',
 'has_morph',
 'has_vector',
 'head',
 'i',
 'idx',
 'iob_strings',
 'is_alpha',
 'is_ancestor',
 'is_ascii',
 'is_bracket',
 'is_currency',
 'is_digit',
 'is_left_punct',
 'is_lower',
 'is_oov',
 'is_punct',
 'is_quote',
 'is_right_punct',
 'is_sent_end',
 'is_sent_start',
 'is_space',
 'is_stop',
 'is_title',
 'is_upper',
 'lang',
 'lang_',
 'le

Checking types confirms: `nlp` is a `spacy.lang.en.English` pipeline object, and `doc` is a `spacy.tokens.doc.Doc`. Understanding these types is essential when extending spaCy with custom components.

In [36]:
type(nlp)

spacy.lang.en.English

In [37]:
type(doc)

spacy.tokens.doc.Doc

###  Span Objects — Slicing a Doc
A `Span` is a slice of a `Doc` (a contiguous sequence of tokens). Use Python list-slice notation `doc[start:end]`. Spans are useful for extracting named entities, noun chunks, or sentence fragments.

In [38]:
span = doc[0:5]
span

Dr. Strange loves pav bhaji

In [39]:
type(span)

spacy.tokens.span.Span

###  Token Attributes — Boolean Flags
spaCy tokens carry boolean flags that allow rule-based filtering. Key flags:
- `is_alpha` — True if all characters are alphabetic
- `is_digit` — True if all characters are digits
- `like_num` — True for number-like tokens (words like `"two"` also return True)
- `is_currency` — True for currency symbols (`$`, `€`, etc.)
- `is_punct` — True for punctuation

Note: `"two".is_digit` is **False** (not digits), but `"two".like_num` is **True** (semantically numeric).

In [40]:
doc = nlp("Tony gave two $ to Peter.")

In [41]:
token0 = doc[0]
token0

Tony

In [42]:
token0.is_alpha

True

In [43]:
token2 = doc[2]
token2

two

In [44]:
token2.is_digit

False

In [45]:
token2.is_alpha

True

In [46]:
token2.like_num

True

In [47]:
token3 = doc[3]
token3

$

In [48]:
type(token3)

spacy.tokens.token.Token

In [49]:
token2.is_alpha

True

In [50]:
token3.is_currency

True

Printing all token attributes in a formatted table helps you see the interplay of flags. Notice: `$` is `is_currency=True` and `is_alpha=False`; `two` is `like_num=True` even though `is_digit=False`.

In [51]:
for token in doc:
    print(token, "==>", "index: ", token.i, "is_alpha:", token.is_alpha, 
          "is_punct:", token.is_punct, 
          "like_num:", token.like_num,
          "is_currency:", token.is_currency,
         )

Tony ==> index:  0 is_alpha: True is_punct: False like_num: False is_currency: False
gave ==> index:  1 is_alpha: True is_punct: False like_num: False is_currency: False
two ==> index:  2 is_alpha: True is_punct: False like_num: True is_currency: False
$ ==> index:  3 is_alpha: False is_punct: False like_num: False is_currency: True
to ==> index:  4 is_alpha: True is_punct: False like_num: False is_currency: False
Peter ==> index:  5 is_alpha: True is_punct: False like_num: False is_currency: False
. ==> index:  6 is_alpha: False is_punct: True like_num: False is_currency: False


###  Real-World Use Case: Extracting Emails from a Student File
Using `token.like_email`, we can scan any unstructured text and extract email addresses — no regex needed. spaCy's tokenizer intelligently keeps `virat@kohli.com` as a single token.

In [52]:
with open("students.txt") as f:
    text = f.readlines()
text

['Dayton high school, 8th grade students information\n',
 '==================================================\n',
 '\n',
 'Name\tbirth day   \temail\n',
 '-----\t------------\t------\n',
 'Virat   5 June, 1882    virat@kohli.com\n',
 'Maria\t12 April, 2001  maria@sharapova.com\n',
 'Serena  24 June, 1998   serena@williams.com \n',
 'Joe      1 May, 1997    joe@root.com\n',
 '\n',
 '\n',
 '\n']

Lines are joined into a single string with spaces — spaCy processes a single `Doc`, so all text must be one string. The `\n` characters inside are handled gracefully by the tokenizer.

In [53]:
text = " ".join(text)
text

'Dayton high school, 8th grade students information\n ==================================================\n \n Name\tbirth day   \temail\n -----\t------------\t------\n Virat   5 June, 1882    virat@kohli.com\n Maria\t12 April, 2001  maria@sharapova.com\n Serena  24 June, 1998   serena@williams.com \n Joe      1 May, 1997    joe@root.com\n \n \n \n'

`token.like_email` returns `True` for any token matching the email pattern (`word@domain.tld`). List comprehension collects only those tokens — a clean, readable approach for information extraction.

In [54]:
doc = nlp(text)
emails = []

for tokn in doc:
    if tokn.like_email:
        emails.append(tokn.text)

emails

['virat@kohli.com',
 'maria@sharapova.com',
 'serena@williams.com',
 'joe@root.com']

###  Customizing the Tokenizer — Special Cases
`nlp.tokenizer.add_special_case()` lets you override how a specific word is tokenized. Here, `"gimme"` → `["gim", "me"]`. The ORTH rule ensures the original text is preserved when rebuilding the string.

In [55]:
from spacy.symbols import ORTH
nlp = spacy.blank("en")

doc = nlp("gimme double cheese extra large healthy pizza")
tokens = [token.text for token in doc]
tokens

['gimme', 'double', 'cheese', 'extra', 'large', 'healthy', 'pizza']

After adding the special case, `"gimme"` is split into `["gim", "me"]`. This is useful for domain-specific contractions (e.g., `"gonna"`, `"wanna"`, `"lemme"`) that standard rules don't handle.

In [56]:
nlp.tokenizer.add_special_case("gimme", [
    {ORTH: "gim"},
    {ORTH: "me"}             # even if we are using special case that should not entirely change the original text
])

doc = nlp("gimme double cheese extra large healthy pizza")
tokens = [token.text for token in doc]
tokens

['gim', 'me', 'double', 'cheese', 'extra', 'large', 'healthy', 'pizza']

###  Sentence Tokenization (Segmentation) — Requires a Pipeline Component
`doc.sents` requires sentence boundaries to be set. With `spacy.blank()`, no parser exists to detect boundaries — you must add the `sentencizer` component explicitly.

 **Expected Error:** The blank pipeline has no component that sets sentence boundaries (`is_sent_start`). Running `doc.sents` throws a `ValueError`. Fix: add `'sentencizer'` to the pipeline.

In [58]:
nlp = spacy.load("en_core_web_sm")   # parser component sets boundaries automatically

doc = nlp("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")
for sentence in doc.sents:
    print(sentence)


Dr. Strange loves pav bhaji of mumbai.
Hulk loves chat of delhi


`nlp.pipeline` returns `[]` — confirming the blank model has no components loaded. This is why `doc.sents` fails above.

In [59]:
nlp.pipeline

[('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x102a0f2e0>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x102a0f880>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x1281b2f10>),
 ('attribute_ruler',
  <spacy.pipeline.attributeruler.AttributeRuler at 0x13ca7b5c0>),
 ('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer at 0x13ca251c0>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x13ae28430>)]

`nlp.add_pipe('sentencizer')` adds a rule-based sentence boundary detector. The `Sentencizer` splits on sentence-ending punctuation (`.`, `!`, `?`). Once added, `doc.sents` works correctly.

In [60]:
nlp.add_pipe('sentencizer')

After adding the sentencizer, `doc.sents` correctly yields 2 sentence spans. The text must be re-processed through `nlp()` to apply the new component.

In [61]:
doc = nlp("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")
for sentence in doc.sents:
    print(sentence)

Dr. Strange loves pav bhaji of mumbai.
Hulk loves chat of delhi
